In [3]:
from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase
from openai import OpenAI
import os
import json

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)


In [5]:
# Define search tool
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

########
# Tell the model about the functions, how to call it and its use.
search_tool = {
    "type": "function",
    "name": "search",
    # So the model knows this tool's use.
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"], # So the model always fill this argument
        "additionalProperties": False
    }
}

In [6]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

# Send the message, and include the tool in the request
response = openai_client.responses.create(
    model = 'llama-3.1-8b-instant',
    input = messages,
    tools = [search_tool],
)

# Instead of a message with answer, the response should contain a
# ResponseFunctionToolCall entry, the model decided that it needs to search the FAQ
# for the neccessary information/context to answer the question.
# Also, in the arguments='{"query":"course join policy"}', the model 
# specified the query it wants to search for in the FAQ.
response.output

[ResponseReasoningItem(id='resp_01kvppcwzmeppazrnp0qezxa9b', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course join policy"}', call_id='ybmmcyw3d', name='search', type='function_call', id='ybmmcyw3d', namespace=None, status='completed')]

In [ ]:
import json

# The second entry is the ResponseFunctionToolCall
call = response.output[1]
# The call.arguments is a JSON string, we need to parse it to get the arguments for the search function
args = json.loads(call.arguments)
# **args means that the "Course join policy" will be parsed into 'query' argument.
results = search(**args)
result_json = json.dumps(results, indent=2)

In [ ]:
# Since the LLMs are stateless between API calls, 
# We need to send the FULL HISTORY of the conversation:
# The question:
messages
# The decision to call search:
messages.extend(response.output)
# The result from following the search tool call:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id, # Link to the specific function call requested.
    "output": result_json,
})

# We call the API a second time with the expanded history:
response = openai_client.responses.create(
    model = 'llama-3.1-8b-instant',
    input = messages,
    tools = [search_tool],
)

response.output_text

# Once the RAG is fully agentic, there will be possibly more round-trips of 
# the model deciding to call more tools, results coming back,
# until the model thinks it has enough information to answer.

# This pattern can be referred to as "agentic RAG", "tool use", or "function calling"
# The LLm decides which tools to call and when.

'Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

# Now, we attempt to make this agentic, by adding this into an LLM loop.

In [ ]:

# 3 parts: intrsuctions, tools, and memory/history.

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

# A function call helper
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

# Processing one response
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model = 'llama-3.1-8b-instant',
    input = messages,
    tools = [search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

# has_function_calls indicates whether the [messages] has new function call
# (because has_function_calls only looks at the last iteration's reponse
# ignoring the previous history that might has previous function calls).
# If TRUE, the messages has new make_call() output that the LLM hasn't seen.
# we need to send the updates [messages] back to the LLM.

NameError: name 'openai_client' is not defined

# The full agent loop

In [ ]:
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model = 'llama-3.1-8b-instant',
        input = messages,
        tools = [search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

# Wrapping all up in a function

In [9]:
def agent_loop(instructions, question, model='llama-3.1-8b-instant') -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [10]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"run Olama locally"}
iteration #2...
ASSISTANT:
To run Ollama locally, you need to install it first by visiting <https://ollama.com/download> and choosing your operating system (macOS, Windows, or Linux). After installation, you can start the Ollama local server by running the command `ollama run llama3` in the terminal. This will download the LLaMA 3 model and start the server. To test the server, you can run the command `curl http://localhost:11434`. Then, you can install the Python client with `pip install ollama` and use it to interact with the local Ollama server.

To run the course locally, you can follow the instructions in the first result. It suggests setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module. Make sure to document your setup and keep your environment reproducible.


'To run Ollama locally, you need to install it first by visiting <https://ollama.com/download> and choosing your operating system (macOS, Windows, or Linux). After installation, you can start the Ollama local server by running the command `ollama run llama3` in the terminal. This will download the LLaMA 3 model and start the server. To test the server, you can run the command `curl http://localhost:11434`. Then, you can install the Python client with `pip install ollama` and use it to interact with the local Ollama server.\n\nTo run the course locally, you can follow the instructions in the first result. It suggests setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module. Make sure to document your setup and keep your environment reproducible.'

# Guardrail for off-topic questions

In [18]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
ASSISTANT:
<brave_search>Queen's Gambit is a chess opening that starts with the moves 1.d4 d5 2.C4.


"<brave_search>Queen's Gambit is a chess opening that starts with the moves 1.d4 d5 2.C4."

In [19]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen Gambit"}
iteration #2...
ASSISTANT:
I was unable to find any results for the question requested.


'I was unable to find any results for the question requested.'

# Next, we want to use a Framework to handle the loop instead of coding the loop ourshelves.

Using ToyAIKit


In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback